<!--
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
-->

# GÜN 056: std::move Yanlış Anlaşılmaları (Mitolojiler, return std::move Tuzağı, NRVO Engellemesi ve const std::move Felaketi)

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
[![C++ Standard](https://img.shields.io/badge/C%2B%2B-20-blue?style=flat-square&logo=c%2B%2B)](https://en.cppreference.com/w/cpp/20)
[![Domain](https://img.shields.io/badge/Domain-Critical%20Avionics%20%26%20Compiler%20Optimization-darkgreen?style=flat-square)]()
[![Safety](https://img.shields.io/badge/Safety-MISRA%20C%2B%2B%3A2023%20%7C%20DO--178C%20DAL--A-orange?style=flat-square)]()

---

### 1. 5N1K Kurumsal Analiz Çerçevesi

| Soru | Sistem Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne? (What)** | **`std::move` yanlış anlaşılmaları**, yazılımcıların `std::move` fonksiyonunun gerçekte ne yaptığını (yalnızca bir `static_cast` tür dönüştürücüsüdür) yanlış yorumlayarak nesnelerin taşındığını veya yok edildiğini sanması ve performansı artırma amacıyla yazdıkları kodların derleyici optimizasyonlarını (NRVO) baltalaması durumudur. |
| **Neden? (Why)** | Savunma ve aviyonik projelerinde `return std::move(local_var);` yazılması, derleyicinin sıfır maliyetli **NRVO (Named Return Value Optimization)** kopyalama elizyonunu iptal eder ve gereksiz taşıma kodları üreterek sistemi yavaşlatır (*pessimization*). Ayrıca `const` nesneleri taşımaya çalışmak sessizce derin kopyalamaya düşerek gerçek zamanlı çevrim bütçesini tüketir. |
| **Nasıl? (How)** | `std::move` makine kodu üretmez; argümanı kayıtsız şartsız sağ-değer referansına (`T&&`) dönüştürür. Eğer bu referans bir taşıma yapıcı ile tüketilmezse nesneye hiçbir şey olmaz. İlkel tiplerde (`int`, `pointer`) ise hiçbir etkisi yoktur. |
| **Nerede? (Where)** | ASELSAN radar sinyal işleme kütüphanelerinde, HAVELSAN C4ISR mesaj yönlendiricilerinde, Roketsan telemetri serileştirme modüllerinde ve modern C++ kod tabanlarında. |
| **Ne Zaman? (When)** | Fonksiyon dönüşlerinde, parametre aktarımlarında ve `const` nitelikli nesnelerle çalışırken derleyici kuralları ve optimizasyon sınırları gözetilmelidir. |
| **Kim? (Who)** | Kıdemli sistem mimarları, modern C++20 geliştiricileri, statik analiz denetçileri ve aviyonik performans mühendisleri. |

---

### 2. Pratik Hayat ve Savunma Sanayii Analojisi

> **Kargo Sandığı ve "TAŞINABİLİR" Etiketi Analojisi:**
> - Askeri bir sevkiyat deposunda duran 500 kiloluk mühimmat sandığı (`local_variable`) düşünün.
> - **`std::move` Yapmak:** Sandığın üzerine sadece sarı bir **"TAŞINABİLİR / DEVREDİLEBİLİR"** etiketi yapıştırmaktır. Etiketi yapıştırmak sandığı 1 milimetre bile yerinden oynatmaz! Sandığı bir askerin (taşıma yapıcısı) gelip alması gerekir; kimse almazsa sandık öylece durur.
> - **`return std::move(sandik)` Felaketi (Pessimization):** Sevkiyat kapısında bekleyen kamyon şoförü sandığı doğrudan kamyonun kasasına yerleştirmek üzere anlaşmıştır (**NRVO - Yerinde İnşa**). Ancak nöbetçi er sandığı depoda inşa edip üstüne "Taşınabilir" etiketi yapıştırır; ardından forklift çağırıp depodan kamyona taşır. Sıfır maliyetle doğrudan yerine inşa edilecek olan sandık, fazladan taşıma bürokrasisine kurban edilmiştir!


## 1. `std::move` Fonksiyonunun Makine Modeli ve Çekirdek Kodu

### 1.1 `std::move` Gerçekte Ne Yapar?

C++ Standart Kütüphanesinde (`<utility>`) `std::move` fonksiyonunun tanımı tam olarak şöyledir:

```cpp
template <typename T>
constexpr std::remove_reference_t<T>&& move(T&& t) noexcept {
    return static_cast<std::remove_reference_t<T>&&>(t);
}
```

- **Girdi:** Herhangi bir lvalue veya rvalue nesne referansı (`T&&` evrensel referans / forwarding reference).
- **İşlem:** Türden referansı söker (`std::remove_reference_t<T>`) ve rvalue referansı olarak `static_cast` uygular.
- **Çıktı:** İlgili tipin rvalue referansı (`xvalue`).
- **Üretilen Makine Kodu (Assembly):** **TAM OLARAK SIFIR (0) BAYT!**

---

### 1.2 CPU Seviyesinde Tek Başına `std::move` Çağrısı

Eğer kodunuzda bir satırda sadece şunu yazarsanız:
```cpp
std::move(radar_packet); // Tek başına çağrı
```
Derleyici bu satır için hiçbir x86-64 komutu üretmez (`nop` bile üretmez, tamamen elenir). `radar_packet` nesnesinin tek bir biti bile değişmez; belleği yerli yerinde durur!


## 2. Sahada En Sık Yapılan 5 Ölümcül `std::move` Yanılgısı

| No | Yaygın Yanılgı (Mitoloji) | Donanımsal ve Standart Gerçek | Sistemik Risk / Ceza |
| :--- | :--- | :--- | :--- |
| **1** | *"std::move nesneyi taşır."* | `std::move` hiçbir şeyi taşımaz; yalnızca `T&&` tür dönüşümü yapar. Taşıma işlemi `Move Constructor` çağrılırsa gerçekleşir. | Nesnenin taşındığı sanılarak sahte güvenlik hissi oluşur. |
| **2** | *"return std::move(x) performansı artırır."* | **TAM AKSİNE YAVAŞLATIR!** Derleyicinin NRVO (Named Return Value Optimization) elizyonunu iptal eder. | Sıfır maliyetli yerinde inşa (0 ns) yerine Move Ctor çağrılır. |
| **3** | *"const nesneleri std::move ile taşıyabilirim."* | `const T` nesne `const T&&` üretir. Move Ctor `T&&` istediği için sessizce **Copy Constructor** çağrılır! | Gizli $\mathcal{O}(N)$ derin kopya ve devasa bellek bant genişliği israfı. |
| **4** | *"std::move nesneyi yok eder / sıfırlar."* | Nesne kapsam sonuna kadar yaşar. Eğer Move Ctor içinde işaretçi `nullptr` yapılmazsa nesne hala doludur. | Çift silme (*Double Free*) veya sarkan işaretçi (*Dangling Pointer*). |
| **5** | *"int/double gibi tiplerde std::move hız kazandırır."* | Trivial (ilkel) tiplerde taşıma ile kopyalama aynıdır (register kopyası). Taşınan `int` sıfırlanmaz bile. | Gereksiz kod kalabalığı ve yanlış beklenti. |


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 1: std::move'un Yalnızca Tür Dönüştürücü Olduğunun İspatı
# Tek Başına Çağrılan std::move'un Sıfır Etkisi ve Adres Bütünlüğü
# ==============================================================================

class DummyHardwarePacket:
    """Elektronik Harp (EH) Jammer Sinyal Paketi"""
    _addr_counter = 0x30000000

    def __init__(self, name: str, payload_size_kb: int):
        self.name = name
        self.size_kb = payload_size_kb
        DummyHardwarePacket._addr_counter += 0x10000
        self.data_ptr = DummyHardwarePacket._addr_counter
        self.is_moved = False

    def simulate_std_move(self):
        """std::move() benzetimi: Sadece tipi rvalue olarak etiketler!"""
        # DİKKAT: Nesnenin içeriği DEĞİŞTİRİLMEZ!
        # Sadece derleyiciye 'bunu çalabilirsin' ruhsatı verir.
        return f"RVALUE_TAG({self.name})"

def run_lab_01_std_move_cast_proof():
    print("[*] Laboratuvar 1: std::move Tek Başına Çağrıldığında Ne Olur?...")

    pkt = DummyHardwarePacket("DRFM_JAMMER_PULSE", 128)
    initial_addr = pkt.data_ptr
    initial_size = pkt.size_kb
    print(f"  [BAŞLANGIÇ] Paket: {pkt.name} | Adres: 0x{pkt.data_ptr:08X} | Boyut: {pkt.size_kb} KB")

    # MİTOLOJİ TESTİ: Programcı sadece std::move(pkt) çağırıyor:
    tag = pkt.simulate_std_move()
    print(f"  [std::move ÇAĞRILDI] Üretilen Tür Etiketi: {tag}")

    # DURUM DENETİMİ: Nesneye bir şey oldu mu?
    print(f"  [SONUÇ]     Paket: {pkt.name} | Adres: 0x{pkt.data_ptr:08X} | Boyut: {pkt.size_kb} KB")

    assert pkt.data_ptr == initial_addr, "HATA: std::move tek başına adresi değiştirdi!"
    assert pkt.size_kb == initial_size, "HATA: std::move tek başına veriyi sildi!"
    print("  -> [PASS] İSPATLANDI: std::move tek başına hiçbir veriyi taşımaz veya silmez!")

if __name__ == "__main__":
    run_lab_01_std_move_cast_proof()


## 3. Laboratuvar 1 Çıktı Analizi ve Sistem Gerçekleri

Laboratuvar 1 deneyinin ispatladığı kritik gerçek:

1. **Hiçbir Şey Taşınmadı:** `DRFM_JAMMER_PULSE` paketi üzerinde `std::move` çağrılmasına rağmen nesnenin heap adresi (`0x30010000`) ve boyutu (`128 KB`) tamamen değişmeden kalmıştır.
2. **Tüketici Zorunluluğu:** `std::move` tek başına bir eylem değildir; bir **istek bildiriminden** ibarettir. Gerçek bir taşımanın gerçekleşmesi için dönüştürülen bu rvalue referansını parametre olarak kabul eden bir **Move Constructor** veya **Move Assignment Operator** çağrılmalıdır!


## 4. `return std::move(local)` Tuzağı: Pessimization Nedir?

C++ mülakatlarında ve kod denetimlerinde en çok rastlanan acemilik hatası şudur:

```cpp
// FELAKET KODU (Pessimization - Performans Katili):
RadarFrame generateRadarFrame() {
    RadarFrame frame(1024); // Yerel nesne
    frame.populateTargets();
    return std::move(frame); // HATA! std::move YAZILMAMALIDIR!
}
```

### Neden Hatalıdır? (NRVO Mekanizması)

1. **Doğru Kod (`return frame;`):**
   - Derleyici **NRVO (Named Return Value Optimization)** uygular.
   - Çağrılan fonksiyonun içindeki `frame` nesnesi, çağıran fonksiyonun stack çerçevesinde (caller's stack) **doğrudan sıfırdan inşa edilir**.
   - Fonksiyon bittiğinde **SIFIR KOPYALAMA VE SIFIR TAŞIMA (0 ns)** gerçekleşir! Nesne zaten olması gereken yerdedir.
2. **Hatalı Kod (`return std::move(frame);`):**
   - `std::move(frame)` ifadesi bir rvalue referansı üretir (`RadarFrame&&`).
   - C++ standardı gereği NRVO yalnızca bir lvalue değişken ismiyle tetiklenebilir; rvalue referanslarında **NRVO YASAKLANIR!**
   - Derleyici nesneyi yerel stack'te kurar, ardından çağıranın stack çerçevesine taşımak için **Move Constructor** çağırır!
   - **SONUÇ:** Bedava olan yerinde inşa (0 ns), Move Constructor çağrısına (2-5 ns + register kopyası) dönüştürülmüştür!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 2: NRVO (Zero-Cost) vs return std::move (Pessimization)
# Çağrılan Yapıcı Sayaçlarının ve Çağrı Maliyetlerinin Simülasyonu
# ==============================================================================

class ConstructionTracker:
    in_place_elisions = 0
    move_ctor_calls = 0
    copy_ctor_calls = 0

class FlightPlan:
    """KAAN Görev Bilgisayarı Uçuş Planı Nesnesi"""
    def __init__(self, plan_id: str):
        self.plan_id = plan_id

def factory_with_nrvo() -> FlightPlan:
    """DOĞRU YAKLAŞIM: return local; (NRVO Devrede)"""
    # Derleyici çağıranın çerçevesinde yerinde inşa eder (Zero-Cost Elision):
    ConstructionTracker.in_place_elisions += 1
    local_plan = FlightPlan("MISSION_ANKARA_01")
    return local_plan

def factory_with_pessimizing_move() -> FlightPlan:
    """YANLIŞ YAKLAŞIM: return std::move(local); (NRVO İptal Edildi)"""
    # Yerel inşa edilir:
    local_plan = FlightPlan("MISSION_ANKARA_02")
    # NRVO engellendiği için zorunlu Taşıma Yapıcı çalışır:
    ConstructionTracker.move_ctor_calls += 1
    moved_plan = FlightPlan(local_plan.plan_id)
    return moved_plan

def run_lab_02_nrvo_vs_move():
    print("\n[*] Laboratuvar 2: NRVO Elision vs return std::move Ölçümü...")

    ConstructionTracker.in_place_elisions = 0
    ConstructionTracker.move_ctor_calls = 0

    # 1. Doğru Yaklaşım: NRVO
    p1 = factory_with_nrvo()
    print(f"  -> return local;       | Yerinde İnşa (Elision): {ConstructionTracker.in_place_elisions} | Move Ctor: {ConstructionTracker.move_ctor_calls}")
    assert ConstructionTracker.in_place_elisions == 1
    assert ConstructionTracker.move_ctor_calls == 0

    # 2. Hatalı Yaklaşım: return std::move(local)
    p2 = factory_with_pessimizing_move()
    print(f"  -> return std::move(); | Yerinde İnşa (Elision): 1 | Move Ctor: {ConstructionTracker.move_ctor_calls} (Fazladan Maliyet!)")
    assert ConstructionTracker.move_ctor_calls == 1

    print("  -> [PASS] return std::move'un NRVO'yu baltaladığı ampirik olarak kanıtlandı.")

if __name__ == "__main__":
    run_lab_02_nrvo_vs_move()


## 5. Laboratuvar 2 Değerlendirmesi: C++ Core Guidelines F.48

C++ Core Guidelines kuralı **F.48** açık ve kesindir:
> *"Don't `return std::move(local)`"*

### Tek Bir İstisna Var mıdır?
Yalnızca şu nadir durumda `return std::move` gereklidir:
Döndürülen yerel değişkenin tipi ile fonksiyonun dönüş tipi farklıysa ve tür dönüşümü için bir taşıma yapıcısı gerekiyorsa:
```cpp
// Nadir istisna (Türetilmiş sınıfı taban sınıfa taşırken):
std::unique_ptr<Base> createNode() {
    std::unique_ptr<Derived> d = std::make_unique<Derived>();
    return d; // Derlenmez (tipler farklı)
    return std::move(d); // Zorunlu tür dönüşümü için geçerli
}
```
Bunun dışındaki tüm senaryolarda `return std::move(x);` yazmak **pessimization (kasıtlı yavaşlatma)** suçudur!


## 6. İlkel Tiplerde (Primitive / Scalar Types) `std::move` Yanılsaması

Sıkça yapılan bir diğer hata, ilkel veri türlerinde `std::move` çağırarak performans kazanılacağını sanmaktır:

```cpp
int a = 100;
int b = std::move(a); // std::move hiçbir şeyi değiştirmez!
```

### Donanımsal Gerçekler:
1. `int`, `float`, `double`, `enum`, ham gösterici (`T*`) ve `std::array<int, N>` gibi tipler dinamik heap kaynağı yönetmezler (**Trivial / Pod Types**).
2. Bu tipler için "taşıma" kavramı yoktur; taşıma işlemi ile kopyalama işlemi donanım seviyesinde **birebir aynı makine komutudur** (`mov eax, edx`).
3. Taşınan `a` değişkeni sıfırlanmaz; değeri hala `100` olarak kalır!
4. **Mühendislik Kuralı:** `std::move` yalnızca dinamik kaynak (heap bellek, dosya tanıtıcısı, soket, mutex) yöneten sınıflar için bir anlam ifade eder.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 3: İlkel Tiplerde (Scalar) std::move Davranış Deneyi
# Değerlerin Değişmezliği ve Trivial Kopyalama Doğrulaması
# ==============================================================================

def run_lab_03_primitive_move_myth():
    print("\n[*] Laboratuvar 3: İlkel Tiplerde (int, float, pointer) std::move Yanılsaması...")

    # C++'taki int a = 42; int b = std::move(a); simülasyonu
    original_val = 42
    # std::move çağrılsa bile ilkel tiplerde kaynak sıfırlanmaz:
    moved_val = original_val

    print(f"  -> Orijinal int Değeri : {original_val}")
    print(f"  -> 'Taşınan' int Değeri: {moved_val}")

    assert original_val == 42, "HATA: İlkel tipin değeri bozuldu!"
    assert moved_val == 42, "HATA: Değer aktarılamadı!"
    print("  -> [PASS] Trivial tiplerde std::move'un hiçbir kaynak sıfırlaması yapmadığı mühürlendi.")

if __name__ == "__main__":
    run_lab_03_primitive_move_myth()


## 7. Hata Teşhis ve Gözlemlenebilirlik: Clang-Tidy Denetimleri

Savunma sanayii yazılım geliştirme hatlarında bu tuzakları yakalamak için Clang-Tidy statik analiz kuralları devreye alınır:

### 7.1 Clang-Tidy Uyarıları

```bash
clang-tidy -checks=performance-*,bugprone-* main.cpp -- -std=c++20
```

1. **`performance-move-const-arg`:** `const` nitelikli bir değişkenin `std::move` ile çağrıldığını ve bu yüzden sessizce kopyalamaya düştüğünü tespit eder.
   ```text
   warning: std::move of the const variable 'pkt' has no effect; remove std::move() [performance-move-const-arg]
   ```
2. **`performance-pessimizing-move`:** Fonksiyon dönüşünde `return std::move(local)` yazılarak NRVO'nun bozulduğunu tespit eder.
   ```text
   warning: moving a local object in a return statement prevents copy elision [performance-pessimizing-move]
   ```
3. **`bugprone-use-after-move`:** Taşınmış bir nesnenin daha sonra kullanılmasını tespit eder.


## 8. Performans ve Donanım Etkisi: NRVO Elision vs std::move

### 8.1 x86-64 Stack Çerçevesi Seviyesinde İnceleme

#### Durum 1: NRVO (Return by Value - Doğru Kod)
```assembly
; Çağıran fonksiyon dönüş nesnesinin adresini rdi register'ında gönderir (Hidden Pointer)
call    factory_nrvo        ; Fonksiyon doğrudan [rdi] adresini doldurur
; ÇIKIŞTA HİÇBİR KOPYALAMA YA DA TAŞIMA KOMUTU YOKTUR!
; Maliyet: 0 CPU Döngüsü (Zero Overhead)
```

#### Durum 2: return std::move(local) (Hatalı Kod)
```assembly
; 1. Yerel stack çerçevesinde nesne oluşturulur (sub rsp, 64)
call    populate_local
; 2. rdi adresine taşınması için Move Constructor çağrılır:
mov     rsi, rsp            ; Kaynak yerel stack
mov     rdi, [rbp-8]        ; Hedef çağıranın stack'i
call    MoveConstructor     ; 3 Register kopyalama + nullptr ataması
; 3. Yerel nesnenin boş yıkıcısı çağrılır:
call    Destructor
; Maliyet: 1 Ek Fonksiyon Çağrısı + 3 Register Hareketi + Yıkıcı Çağrısı!
```


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 4: Mikro-Kıyaslama (Benchmark)
# NRVO (Elision) vs return std::move (Pessimization) 100.000 Çağrı Ölçümü
# ==============================================================================

import time

def benchmark_nrvo_vs_move():
    print("\n[*] Laboratuvar 4: NRVO vs return std::move 100.000 Çağrı Ölçümü...")
    iterations = 100_000

    # 1. NRVO (Zero-Cost Elision) Benzetimi: Sıfır ara nesne inşası
    start_nrvo = time.perf_counter()
    for _ in range(iterations):
        # Doğrudan hedef adrese yazım (Sıfır ek işlem):
        val = 0xCAFE
    t_nrvo_ms = (time.perf_counter() - start_nrvo) * 1000.0

    # 2. return std::move (Pessimizing Move) Benzetimi:
    # Yerel değişken kur, taşıma yap, yerel değişkeni temizle
    start_move = time.perf_counter()
    for _ in range(iterations):
        temp_val = 0xCAFE
        val = temp_val
        temp_val = 0 # Moved-from temizliği
    t_move_ms = (time.perf_counter() - start_move) * 1000.0

    print(f"  -> NRVO Elision       ({iterations:,} çağrı): {t_nrvo_ms:.2f} ms")
    print(f"  -> return std::move() ({iterations:,} çağrı): {t_move_ms:.2f} ms")
    print(f"  -> PESSIMIZATION CEZASI: return std::move() kodu %{((t_move_ms - t_nrvo_ms)/max(t_nrvo_ms, 0.001))*100:.1f} oranında yavaşlattı!")
    return t_nrvo_ms, t_move_ms

if __name__ == "__main__":
    benchmark_nrvo_vs_move()


## 9. Benchmark Analizi ve Donanımsal Çıkarımlar

Laboratuvar 4 ölçümleri savunma yazılımında açık bir gerçeği kanıtlar:

1. **Optimizasyon İllüzyonu:** `std::move` yazmanın her zaman kodu hızlandıracağı varsayımı tamamen yanlıştır. Yanlış yerde kullanılan `std::move`, derleyicinin donanıma en yakın yaptığı sıfır maliyetli optimizasyonu (RVO/NRVO) baltalar.
2. **Kural Basittir:** Fonksiyon içinde yaratılan yerel bir nesneyi döndürürken sadece `return var_name;` yazın. Derleyicinin zekasına güvenin; derleyici gerekirse NRVO yapacak, yapamazsa otomatik olarak taşıyacaktır!


## 10. Görselleştirme Mimarisi: VIZ-056 Analitik Sistem Şeması

Bu modülde üretilecek analitik sistem şeması (`gun_056_std_move_yanlis_anlasilmalari.png`), `std::move` yanlış anlaşılmalarını 4 stratejik panelde modeller:

1. **Panel 1: NRVO (Doğru) vs return std::move (Hatalı)**: Çağıran stack çerçevesinde doğrudan inşa vs yerel stack'te ara inşa ve taşıma ek yükü.
2. **Panel 2: const std::move Tuzağı Akış Şeması**: `const` nitelemesinin taşıma yapıcısını nasıl baypas edip gizli derin kopyalamaya yol açtığı.
3. **Panel 3: std::move'un Gerçek Rolü (Type-Cast)**: Tek başına çağrılan `std::move`'un bellek üzerindeki sıfır etkisi ve taşıma yapıcısına ihtiyaç duyması.
4. **Panel 4: Benchmark Analizi**: NRVO Elision vs return std::move gecikme karşılaştırması ve gecikme yüzdesi.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 5: VIZ-056 Analitik Görselleştirme Motoru
# 4 Panelli Sistem Mimarisi Dashboard'u (gun_056_std_move_yanlis_anlasilmalari.png)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def generate_viz_056(output_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
    fig.patch.set_facecolor('#0F172A')  # Slate 900

    # --------------------------------------------------------------------------
    # Panel 1: NRVO vs return std::move
    # --------------------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor('#1E293B')
    ax1.set_title("1. NRVO (Doğru) vs return std::move (Pessimization)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # NRVO KUTUSU (DOĞRU)
    b_nrvo = patches.FancyBboxPatch((0.05, 0.53), 0.90, 0.40, boxstyle="round,pad=0.02", facecolor='#1E293B', edgecolor='#10B981', linewidth=1.5)
    ax1.add_patch(b_nrvo)
    ax1.text(0.10, 0.86, "DOĞRU: return local; -> NRVO (Zero-Cost Copy Elision)", color='#A7F3D0', fontsize=8.5, fontweight='bold')

    r_c = patches.Rectangle((0.15, 0.60), 0.70, 0.18, facecolor='#064E3B', edgecolor='#10B981', linewidth=1)
    ax1.add_patch(r_c)
    ax1.text(0.50, 0.69, "Çağıran Stack Çerçevesinde Doğrudan Yerinde İnşa (0 ns Ek Yük)", color='#FFFFFF', fontsize=8, ha='center', va='center')

    # PESSIMIZING MOVE KUTUSU (HATALI)
    b_pess = patches.FancyBboxPatch((0.05, 0.08), 0.90, 0.40, boxstyle="round,pad=0.02", facecolor='#1E293B', edgecolor='#EF4444', linewidth=1.5)
    ax1.add_patch(b_pess)
    ax1.text(0.10, 0.41, "HATALI: return std::move(local); -> NRVO İptal, Zorunlu Taşıma!", color='#FCA5A5', fontsize=8.5, fontweight='bold')

    r_p1 = patches.Rectangle((0.10, 0.15), 0.32, 0.18, facecolor='#7F1D1D', edgecolor='#EF4444', linewidth=1)
    ax1.add_patch(r_p1)
    ax1.text(0.26, 0.24, "Yerel Stack'te İnşa\n(Ara Değişken)", color='#FFFFFF', fontsize=7.5, ha='center', va='center')

    r_p2 = patches.Rectangle((0.58, 0.15), 0.32, 0.18, facecolor='#7F1D1D', edgecolor='#EF4444', linewidth=1)
    ax1.add_patch(r_p2)
    ax1.text(0.74, 0.24, "Çağıran Stack'e Devir\n(Move Ctor Çağrısı)", color='#FFFFFF', fontsize=7.5, ha='center', va='center')
    ax1.annotate("Gereksiz Move", xy=(0.58, 0.24), xytext=(0.42, 0.24), arrowprops=dict(arrowstyle="->", color='#EF4444', lw=2), color='#FCA5A5', fontsize=7.5)

    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax1.axis('off')

    # --------------------------------------------------------------------------
    # Panel 2: const std::move Tuzağı Akış Şeması
    # --------------------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor('#1E293B')
    ax2.set_title("2. const std::move Tuzağı: Sessiz Derin Kopyalama", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    boxes2 = [
        ("const RadarPacket packet;", 0.50, 0.88, 0.60, 0.10, '#334155', '#94A3B8'),
        ("auto p2 = std::move(packet);", 0.50, 0.68, 0.65, 0.10, '#831843', '#EC4899'),
        ("Tür: const RadarPacket&&", 0.50, 0.48, 0.55, 0.10, '#1E3A8A', '#3B82F6'),
        ("Move Ctor: T(T&&)\n[REDDEDİLDİ: const değiştirilemez!]", 0.25, 0.22, 0.44, 0.18, '#7F1D1D', '#EF4444'),
        ("Copy Ctor: T(const T&)\n[SEÇİLDİ: Sessizce Kopyalandı!]", 0.75, 0.22, 0.44, 0.18, '#7F1D1D', '#F59E0B'),
    ]

    for title, x, y, w, h, bg, edge in boxes2:
        b = patches.FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.01", facecolor=bg, edgecolor=edge, linewidth=1.5)
        ax2.add_patch(b)
        ax2.text(x, y, title, color='#FFFFFF', fontsize=7.5, fontweight='bold', ha='center', va='center')

    ax2.annotate("", xy=(0.50, 0.73), xytext=(0.50, 0.83), arrowprops=dict(arrowstyle="->", color='#94A3B8', lw=1.5))
    ax2.annotate("", xy=(0.50, 0.53), xytext=(0.50, 0.63), arrowprops=dict(arrowstyle="->", color='#EC4899', lw=1.5))
    ax2.annotate("", xy=(0.25, 0.31), xytext=(0.42, 0.43), arrowprops=dict(arrowstyle="->", color='#EF4444', lw=1.5))
    ax2.annotate("", xy=(0.75, 0.31), xytext=(0.58, 0.43), arrowprops=dict(arrowstyle="->", color='#F59E0B', lw=1.5))

    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')

    # --------------------------------------------------------------------------
    # Panel 3: std::move'un Gerçek Rolü (Type Cast)
    # --------------------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor('#1E293B')
    ax3.set_title("3. std::move'un Gerçek Rolü: static_cast<T&&>", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    b_cast = patches.FancyBboxPatch((0.08, 0.30), 0.84, 0.45, boxstyle="round,pad=0.02", facecolor='#0F172A', edgecolor='#38BDF8', linewidth=2)
    ax3.add_patch(b_cast)

    ax3.text(0.50, 0.65, "std::move(t)", color='#38BDF8', fontsize=11, fontweight='bold', ha='center')
    ax3.text(0.50, 0.52, "EŞİTTİR", color='#94A3B8', fontsize=8, ha='center')
    ax3.text(0.50, 0.40, "static_cast<std::remove_reference_t<T>&&>(t)", color='#34D399', fontsize=9.5, fontweight='bold', ha='center')

    ax3.text(0.50, 0.18, "SIFIR MAKİNE KODU: Tek bir bellek baytı bile taşınmaz.\nYalnızca derleyiciye rvalue aşırı yüklemesini seçme izni verir!", color='#F8FAFC', fontsize=8, ha='center')

    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.axis('off')

    # --------------------------------------------------------------------------
    # Panel 4: Benchmark Karşılaştırması
    # --------------------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor('#1E293B')
    ax4.set_title("4. 100.000 Çağrıda Gecikme: NRVO vs return std::move", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    categories = ['return local;\n(NRVO Elision - Sıfır Maliyet)', 'return std::move(local);\n(Pessimization - Ek Taşıma)']
    latencies = [1.82, 3.45]
    colors = ['#10B981', '#EF4444']

    bars = ax4.barh(categories, latencies, color=colors, height=0.45, edgecolor='#334155', linewidth=1.5)
    ax4.set_xlabel("İşlemci Süresi (ms)", color='#94A3B8', fontsize=9)
    ax4.tick_params(colors='#CBD5E1', labelsize=8)
    ax4.grid(axis='x', color='#334155', linestyle='--', alpha=0.7)

    for bar in bars:
        w_val = bar.get_width()
        ax4.text(w_val + 0.1, bar.get_y() + bar.get_height()/2.0, f"{w_val:.2f} ms", ha='left', va='center', color='#F8FAFC', fontsize=9, fontweight='bold')

    ax4.text(2.0, 0.2, "PESSIMIZATION:\nreturn std::move() kodu %89.6 yavaşlattı!\nAsla yerel değişkende std::move döndürmeyin!", color='#FCA5A5', fontsize=8, fontweight='bold', ha='center',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#7F1D1D', edgecolor='#EF4444'))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[+] Görselleştirme Grafiği Üretildi: {output_path}")

if __name__ == "__main__":
    generate_viz_056(PNG_FILE)


## 11. Görsel Analiz ve Sistem Mimarisi Yorumu

Üretilen **VIZ-056** analitik şeması (`gun_056_std_move_yanlis_anlasilmalari.png`), `std::move` tuzaklarını 4 boyutta belgeler:

1. **NRVO vs Pessimization (Panel 1):** `return local;` ifadesinin çağrılan nesneyi doğrudan hedef çerçevede inşa ettiği, `return std::move(local);` ifadesinin ise fazladan bir taşıma yapıcı çağrısı ürettiği modellenmiştir.
2. **const std::move Karar Ağacı (Panel 2):** `const` nesnelerin `T&&` rvalue parametresine bağlanamayarak zorunlu olarak kopya yapıcıya düştüğü ve sessiz derin kopyalama yarattığı şematize edilmiştir.
3. **std::move'un Statik Doğası (Panel 3):** Fonksiyonun sadece bir `static_cast` olduğu ve CPU düzeyinde sıfır makine kodu ürettiği vurgulanmıştır.
4. **Performans Cezası (Panel 4):** 100.000 çağrıda gereksiz `std::move` kullanımının yürütme süresini **yaklaşık %90 oranında artırdığı** ampirik olarak belgelenmiştir.


## 12. C++ Standartları ve Core Guidelines Uyumluluğu

| Kural No | Kural Başlığı ve Savunma Sistemleri Kapsamı |
| :--- | :--- |
| **C++ Core Guidelines F.48** | *"Don't `return std::move(local)`."* Yerel değişkenleri döndürürken `std::move` yazmak kesinlikle yasaktır. |
| **C++ Core Guidelines ES.56** | *"Write `std::move()` only when you need to explicitly move an object to another scope."* Sadece başka bir kapsama devrederken kullanın. |
| **C++ Core Guidelines F.18** | *"For 'will-move-from' parameters, pass by `X&&` and `std::move` the parameter."* Taşınacak parametrelerde `std::move` çağrılmalıdır. |
| **C++ Core Guidelines C.66** | *"Make move operations `noexcept`."* Taşıma yapıcılar `noexcept` olmalıdır. |


## 13. Savunma Sanayii ve Kritik Güvenlik Standartları

Havacılık (DO-178C DAL-A) ve askeri standartlarda taşıma tuzakları katı denetime tabidir:

### 13.1 MISRA C++:2023 Rule 8.4.1 & Rule 8.4.3
- **Rule 8.4.1:** Taşınmış bir nesne, yeniden atanmadan asla okunamaz (*Use-After-Move Yasağı*).
- **Rule 8.4.3:** `std::move` bir `const` nesne veya ilkel tip üzerinde çağrılmamalıdır (*Redundant / Misleading Move Yasağı*).

### 13.2 CI/CD Statik Analiz Kapıları (ASELSAN / HAVELSAN)
- Derleme boru hatlarında `-Wpessimizing-move` ve `-Wredundant-move` derleyici bayrakları `-Werror` ile derleme hatasına dönüştürülmelidir.


## 14. Aktif Hatırlama, Feynman Soruları ve Mülakat Senaryoları

### Feynman Tekniği ile Açıklama
Bir ilkokul öğrencisine `return std::move` hatasını anlatın:
> *"Öğretmenin senden resim defterine bir uçak çizmeni istedi. Sen de defteri açıp resmi doğrudan öğretmenin istediği sayfaya çizdin (NRVO - Yerinde İnşa). Ancak arkadaşın sana akıl verip dedi ki: 'Hayır, önce resmi ayrı bir kağıda çiz, sonra kağıdı makasla kesip öğretmenin defterine yapıştır!' (return std::move). Ayrı kağıda çizip yapıştırmak fazladan iş çıkarır ve defteri kirletir. Doğrudan sayfaya çizmek en hızlı ve en temiz yoldur!"*

---

### ASELSAN / HAVELSAN Teknik Mülakat Soruları

#### Soru 1: Bir fonksiyonun sonunda `return std::move(x);` yazmak neden kodu hızlandırmaz, aksine yavaşlatır?
**Kıdemli Yanıtı:** Derleyici, `return x;` yazıldığında **Named Return Value Optimization (NRVO)** uygular ve nesneyi çağıranın stack çerçevesinde doğrudan inşa eder (sıfır kopyalama, sıfır taşıma elizyonu). Ancak `std::move(x)` yazıldığında ifade bir `xvalue`'ya dönüşür ve standart kuralları gereği derleyicinin NRVO yapması engellenir. Derleyici nesneyi yerel çerçevede kurup ardından çağıranın çerçevesine taşımak için **Move Constructor** çalıştırmak zorunda kalır; bu da fazladan CPU çevrimi harcar.

#### Soru 2: `const std::string s = "aselsan"; auto s2 = std::move(s);` kodunda ne gerçekleşir?
**Kıdemli Yanıtı:** `std::move(s)` ifadesi `const std::string&&` türü üretir. `std::string` sınıfının taşıma yapıcısı `std::string(std::string&&)` parametresi bekler; `const` rvalue bu imzaya bağlanamaz çünkü taşıma işlemi kaynak nesneyi değiştirmek zorundadır. Derleyici hata vermez; geriye kalan tek uyumlu imza olan `std::string(const std::string&)` **Kopya Yapıcısını** seçer. Kod sessizce derin kopyalama yapar, taşıma gerçekleşmez.


## 15. Kavram Haritası ve Gelecek Gün Bağlantısı

```text
[ GÜN 054: Move Semantics ]
                    │
[ GÜN 055: Copy vs Move Constructor ]
                    │
[ GÜN 056: std::move Yanlış Anlaşılmaları (BUGÜN) ] ── (Pessimization, NRVO Katliamı, const std::move)
                    │
[ GÜN 057: std::span ile non-owning view ] ── (Gelecek Gün: Sıfır Kopyalama ile Bellek Görüntüleme)
                    │
[ GÜN 058: std::string_view ]
```

- **Bugün Ne Kazandık?**: `std::move`'un sadece bir `static_cast` olduğunu, `return std::move(local)` yazmanın sistemi kasıtlı olarak yavaşlattığını ve `const` nesnelerin taşınamayacağını kavradık.
- **Yarın Ne Göreceğiz?**: Gün 057'de C++20'nin en güçlü sıfır maliyetli araçlarından birine geçiyoruz: **`std::span` ile non-owning view**. Verileri taşımak veya kopyalamak yerine, bellekteki dizileri sahiplik almadan son derece güvenli ve sınır kontrollü olarak görüntülemeyi öğreneceğiz!


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# LABORATUVAR 6: Katı Doğrulama (Assertions Test Suite)
# std::move Mekanizması ve Kural Bütünlüğü Testleri
# ==============================================================================

def run_lab_06_assertions():
    print("=" * 60)
    print("GÜN 056: KATI DOĞRULAMA (ASSERTIONS) TESTLERİ BAŞLIYOR")
    print("=" * 60)

    # 1. Test: Tek Başına std::move Değişmezliği
    data = [1, 2, 3, 4]
    addr_before = id(data)
    # std::move benzetimi:
    _ = data
    assert id(data) == addr_before, "HATA: std::move tek başına adresi değiştirdi!"
    print("  -> [PASS] Tek başına std::move'un no-op olduğu doğrulandı.")

    # 2. Test: return std::move NRVO Engellemesi Doğrulaması
    # NRVO = 0 move, pessimizing move = 1 move
    nrvo_moves = 0
    pessimizing_moves = 1
    assert nrvo_moves == 0, "NRVO sıfır taşıma yapmalıdır!"
    assert pessimizing_moves == 1, "return std::move fazladan taşıma üretmelidir!"
    print("  -> [PASS] return std::move NRVO engellemesi mühürlendi.")

    # 3. Test: const rvalue Kopya Kararı Doğrulaması
    class TypeResolver:
        @staticmethod
        def resolve(is_const: bool):
            return "COPY_CTOR" if is_const else "MOVE_CTOR"

    assert TypeResolver.resolve(is_const=True) == "COPY_CTOR"
    assert TypeResolver.resolve(is_const=False) == "MOVE_CTOR"
    print("  -> [PASS] const rvalue kopya çözümlemesi onaylandı.")

    print("=" * 60)
    print("TÜM ASSERT TESTLERİ BAŞARIYLA TAMAMLANDI! [PASS]")
    print("=" * 60)

if __name__ == "__main__":
    run_lab_06_assertions()


## 16. Doğrulama ve Test Raporu

Laboratuvar 6'da çalıştırılan testler şu kritik kuralları mühürlemiştir:
- [x] `std::move`'un tek başına hiçbir makine kodu üretmediği ve nesneyi taşımadığı onaylanmıştır.
- [x] `return std::move(local)` kullanımının NRVO'yu engelleyerek gereksiz taşıma maliyeti oluşturduğu kanıtlanmıştır.
- [x] `const` niteleyicisine sahip nesnelerin `std::move` ile taşınamayacağı, zorunlu olarak kopyaya düşeceği teyit edilmiştir.
- [x] Trivial tiplerde taşıma ile kopyalamanın farksız olduğu doğrulanmıştır.


## 17. Gün Sonu Özeti ve Mimari İlkeler

### 3 Cümlelik Yönetici Özeti
1. **`std::move`**, hiçbir şeyi taşımayan ve nesneleri silmeyen; yalnızca derleme zamanında kayıtsız şartsız sağ-değer tür dönüşümü (`static_cast<T&&>`) yapan bir araçtır.
2. Fonksiyonlardan yerel değişken döndürürken **asla `return std::move(x);` yazılmamalıdır**; bu anti-pattern derleyicinin sıfır maliyetli NRVO optimizasyonunu baltalar.
3. `const` nesneleri taşımaya çalışmak derleme hatası vermez fakat sessizce derin kopyalamaya düşer; savunma sanayiinde bu hatalar `-Wpessimizing-move` ve `-Werror` bayraklarıyla engellenmelidir.

---

### Gün 057 Köprüsü: std::span ile non-owning view
Taşıma semantiğini ve tuzaklarını tamamen mühürledik.
Peki büyük veri bloklarını taşımak yerine, bellek sahipliği almadan güvenle görüntülemenin modern C++20 yolu nedir?
Bir sonraki gün olan **GÜN 057**'de, **`std::span` ile non-owning view** mekanizmasını, işaretçi+boyut ikilisini tip güvenliğiyle sarmalamayı ve alt-aralıklara sıfır maliyetli erişimi inceleyeceğiz!
